This file fully prepares data for training. Make sure to back up all your songs before this if you want to play them aswell. Recommended to run in order

Move all folders to top level ##note this needs to be refined, ESPICALLLY THE CHECK FOR DUPLICATE FOLDER

In [7]:
import os
from pathlib import Path


def delete_empty_folders(root_folder):
    for dirpath, dirnames, files in os.walk(root_folder, topdown=False):
        #Some song packs have unneeded information in .txts
        for f in files:
            if f.endswith('.txt'):
                full_path = os.path.join(dirpath, f)
                try:
                    os.remove(full_path)
                    print(f"Deleted file: {full_path}")
                except Exception as e:
                    print(f"Error deleting file {f}: {e}")


        ##WE CAN USE OS.RMDIR AS OUR CHECK FOR EMPTY DIRECTORYS AS ONLY WORKS IF DIRECTORY IS EMPTY
        try:
            os.rmdir(dirpath)
            print(f"Deleted empty folder: {dirpath}")
        except OSError:
            pass
        except Exception as e:
            print(f"Failed to delete error: {e}")

def move_contents(src_folder, dest_folder):
    """
    Move src_folder to dest_folder

    Parameters:
    src_folder (str): The source folder where contents will be moved from.
    dest_folder (str): The destination folder where contents will be moved to.
    """
    src = Path(src_folder)
    dest = Path(dest_folder)

    # Handle duplicate names to prevent crashing
    target_path = dest / src.name
    if target_path.exists():
        print(f"Duplicate detected. Deleting the one hidden in the folder")
        try:
            os.rmdir(src)
            print(f"removed {src} as is a duplicate")
            return
        except Exception as e:
            print(f"Error deleteing {src}: {e}")

##NORMAL MOVE
    try:
        shutil.move(str(src), str(dest))
        print(f"Moved: {src} -> {dest}")
    except Exception as e:
        print(f"Error moving {src}: {e}")


def process_folders(base_folder):
    """
    Go through each folder till theres no more and then move those folders to the base folder

    Parameters:
    base_folder (str): The base folder which contains the folders to process.
    """


    for dirpath, dirnames, files in os.walk(base_folder, topdown=False):

        #renamed for readability
        current_path = dirpath

        if current_path == base_folder:
            continue

        #if has no subdirectories
        if not dirnames:
            move_contents(dirpath, base_folder)

    #Preform a clean up pass to delete the now fully empty paths
    delete_empty_folders(base_folder)

if __name__ == "__main__":
    #CHANGE THIS MAYBE
    training_path = Path(r'D:\tensor-hero2.0\Training Data') / "Unprocessed"

    if training_path.exists():
        print(f"Processing: {training_path}")
        process_folders(training_path)
    else:
        print(f"Path not found: {training_path}")

Processing: D:\tensor-hero2.0\Training Data\Unprocessed
Duplicate detected. Renaming destination to: 'Weird Al' Yankovic - Virus Alert (ImPhanty)16.51.2814.31.5414.32.08
Moved: D:\tensor-hero2.0\Training Data\Unprocessed\'Weird Al' Yankovic - Virus Alert (ImPhanty)16.51.2814.31.54 -> D:\tensor-hero2.0\Training Data\Unprocessed\'Weird Al' Yankovic - Virus Alert (ImPhanty)16.51.2814.31.5414.32.08
Duplicate detected. Renaming destination to: (FULL ALBUM) Fall Out Boy - American Beauty_American Psycho [Deltarak]14.31.5414.32.08
Moved: D:\tensor-hero2.0\Training Data\Unprocessed\(FULL ALBUM) Fall Out Boy - American Beauty_American Psycho [Deltarak]14.31.54 -> D:\tensor-hero2.0\Training Data\Unprocessed\(FULL ALBUM) Fall Out Boy - American Beauty_American Psycho [Deltarak]14.31.5414.32.08
Duplicate detected. Renaming destination to: .38 Special - Hold on Loosely14.31.5414.32.08
Moved: D:\tensor-hero2.0\Training Data\Unprocessed\.38 Special - Hold on Loosely14.31.54 -> D:\tensor-hero2.0\Train

Search for and convert any songs in opus format

In [25]:
import os
import subprocess

#MAY NEED TO CHANGE DEPENDING ON IF ITS SET AS A PATH VARIABLE OR NOT
FFMPEGPATH = r"D:\ffmpeg-2024-05-06-git-96449cfeae-essentials_build\ffmpeg-2024-05-06-git-96449cfeae-essentials_build\bin\ffmpeg.exe"

# Function to recursively search and delete directories containing .opus files
def delete_folders_with_opus(root_dir):
    for dirpath, dirnames, filenames in os.walk(root_dir, topdown=False):
        for file in filenames:
            if file.endswith('.opus'):
                file_path = os.path.join(dirpath, file)

                #0 grabs the root
                new_name = os.path.splitext(str(file))[0] + ".wav"
                output_path = os.path.join(dirpath, new_name)


                command = [FFMPEGPATH, '-i', str(file_path), '-vn' ,str(output_path)]
                try:
                    result = subprocess.run(command, capture_output=True, text=True)

                    if result.returncode == 0:
                        print(f"Done converting {file_path}")
                        os.remove(file_path)
                    else:
                        print(f"{file} Conversion failed")
                except Exception as e:
                    print(f"An error eccored in converting to wav, {file_path}: {e}")

# Replace 'root_directory_path' with the path to the root directory where you want to start
root_directory_path = Path(r'D:\tensor-hero2.0\Training Data') / "Unprocessed"# os.path.join(os.path.dirname(__file__), "Unprocessed")

delete_folders_with_opus(root_directory_path)

Done converting D:\tensor-hero2.0\Training Data\Unprocessed\'Weird Al' Yankovic - Virus Alert (ImPhanty)\song.opus


Exception in thread Thread-1038:
Traceback (most recent call last):
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\threading.py", line 932, in _bootstrap_inner
    self.run()
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\subprocess.py", line 1386, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 1694: character maps to <undefined>


Done converting D:\tensor-hero2.0\Training Data\Unprocessed\3-UP - Constant Motion (Dream Theater cover) (❮b❯❮color=#ffff33❯3-UP)\3-UP - Constant Motion (Dream Theater cover) (❮b❯❮color=#ffff33❯3-UP)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Albert Ayler - Blues [untitled piece] (Haggis)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Alice Cooper - Feed My Frankenstein\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\America - Ventura Highway (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Animals As Leaders - CAFO\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Astronaut - Witchcraft [Deltarak]\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Basement Jaxx - Red Alert (Jaxx Radio Mix) (Pix_)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Beartooth - Aggressive (Yhughu)\Beartooth - Aggressive (Yhughu)\guitar.opus

Exception in thread Thread-1064:
Traceback (most recent call last):
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\threading.py", line 932, in _bootstrap_inner
    self.run()
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\subprocess.py", line 1386, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 1614: character maps to <undefined>


Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Bomfunk MC's - Freestyler (❮color=#5A5AFF❯Z❮color=#7A7AFF❯e❮color=#9494FF❯z❮color=#CCCCFF❯o)\Bomfunk MC's - Freestyler (❮color=#5A5AFF❯Z❮color=#7A7AFF❯e❮color=#9494FF❯z❮color=#CCCCFF❯o)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Boom Kitty & HeliXiX - Tokyo Takedown [Deltarak]\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Boston - Cool the Engines (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Buckethead - Disceto (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Buckethead - Hog Bitch Stomp\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Buckethead - King James (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Buckethead - Welcome to Bucketheadland (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Bumblefoot -

Exception in thread Thread-1122:
Traceback (most recent call last):
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\threading.py", line 932, in _bootstrap_inner
    self.run()
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\subprocess.py", line 1386, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 1828: character maps to <undefined>


Done converting D:\tensor-hero2.0\Training Data\Unprocessed\DragonForce - Behind the Mirror of Death (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\DragonForce - Heart Demolition (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\DragonForce - In a Skyforged Dream (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\DragonForce - My Heart Will Go On (Celine Dion Cover) (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\DragonForce - Remembrance Day (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\DragonForce - Strangers (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\DragonForce - The Last Dragonborn (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Dragonforce - Through the Fire and Flames (Harmonix)\Dragonforce - Through the Fire and F

Exception in thread Thread-1172:
Traceback (most recent call last):
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\threading.py", line 932, in _bootstrap_inner
    self.run()
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\subprocess.py", line 1386, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 1805: character maps to <undefined>


Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Extreme Power Medley (Full Album)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Fall Out Boy - Stay Frosty Royal Milk Tea [Deltarak]\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\FamilyJules - Long Live the New Fresh (Danimal Cannon cover) [Deltarak]\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Feuerschwanz - Schubsetanz [GuitarZero132]\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Fight The Fury - Lose Hold Of It All\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Five Finger Death Punch - The Way of the Fist (GuitarZero132)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Frank Zappa - Nanook Rubs It\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Getsix - Red Nightmare [Deltarak]\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Greta Van Fleet 

Exception in thread Thread-1324:
Traceback (most recent call last):
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\threading.py", line 932, in _bootstrap_inner
    self.run()
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\subprocess.py", line 1386, in _readerthread
    buffer.append(fh.read())
  File "C:\Users\jethr\.conda\envs\tensor-hero\lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 1668: character maps to <undefined>


Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Santana - Stand Up ／ Runnin (Eva)\Santana - Stand Up ／ Runnin (Eva)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Scorpions - Coming Home\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Seven Spires - Choices [Deltarak]\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Slow Gherkin - Shed Some Skin (Justin？)\Slow Gherkin - Shed Some Skin (Justin？)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Spiritbox - The Void [Deltarak]\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\SpongeBob SquarePants - Gary Come Home (Gary's Song) (Justin？)\song.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Steve Lacy - Dark Red (Picquerist)\Steve Lacy - Dark Red (Picquerist)\guitar.opus
Done converting D:\tensor-hero2.0\Training Data\Unprocessed\Steve Lacy - Dark Red (Picquerist)\guitar.opus
Done converting D:\tensor-hero

Convert midi to clone hero .charts

In [8]:
import subprocess
import os
from pathlib import Path
import shutil

##!! CHANGE THIS POSSIBLY ##!!
keep_midi = True

def run_exe_with_file(exe_path, file_path):
    """
    Runs an executable file, passing the specified file path as a command-line argument.

    Args:
        exe_path (str): The path to the executable (.exe) file.
        file_path (str): The path to the file you want to "drop" onto the exe.
    """

    # Ensure paths are absolute and correctly formatted for the OS
    full_exe_path = os.path.abspath(exe_path)
    full_file_path = os.path.abspath(file_path)

    # The executable receives the file path as the first command-line argument
    command = [full_exe_path, full_file_path, '-m']
    try:
        print(f"Running command: {command}")
        subprocess.run(command, check=True, shell=False)
        print("Executable ran successfully.")
        print("-" * 20)

    except FileNotFoundError:
        print(f"Error: The executable was not found at {full_exe_path}")
    except subprocess.CalledProcessError as e:
        print(f"Error: Executable failed with return code {e.returncode}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


#!!! CHANGE TO THE LOCATION OF YOUR MID2CHART APPLCATION
executable_path = Path(r"D:\mid2chart\mid2chart.exe")


# Function to recursively search and delete directories containing .opus files
def search_for_midi(root_dir):
    for dirpath, dirnames, filenames in os.walk(root_dir, topdown=False):
        for filename in filenames:
            ###!!NEED ADD CHECK FOR IF THERE IS .CHART AND .MIDI AND USE THE OG CHART
            if filename.lower().endswith('.chart'):
                continue
            if filename.lower().endswith(('.midi', '.mid')):

                # Construct the full path to the midi file
                full_file_path = os.path.join(dirpath, filename)

                # Run the exe with this specific file
                run_exe_with_file(executable_path, full_file_path)
                if keep_midi == False:
                    os.remove(full_file_path)

# maybe replace 'root_directory_path' with the path to the root directory where you want the search to start
root_directory_path = Path(r"D:\tensor-hero2.0\Training Data\Unprocessed")


if __name__ == "__main__":
    search_for_midi(root_directory_path)


Running command: ['D:\\mid2chart\\mid2chart.exe', 'D:\\tensor-hero2.0\\Training Data\\Unprocessed\\.38 Special - Hold on Loosely\\notes.mid', '-m']
Error: Executable failed with return code 3762504530
Running command: ['D:\\mid2chart\\mid2chart.exe', 'D:\\tensor-hero2.0\\Training Data\\Unprocessed\\01 Paladin - Awakening\\notes.mid', '-m']
Error: Executable failed with return code 3762504530
Running command: ['D:\\mid2chart\\mid2chart.exe', 'D:\\tensor-hero2.0\\Training Data\\Unprocessed\\02 Paladin - Divine Providence\\notes.mid', '-m']
Error: Executable failed with return code 3762504530
Running command: ['D:\\mid2chart\\mid2chart.exe', 'D:\\tensor-hero2.0\\Training Data\\Unprocessed\\03 Paladin - Carpe Diem\\notes.mid', '-m']
Error: Executable failed with return code 3762504530
Running command: ['D:\\mid2chart\\mid2chart.exe', 'D:\\tensor-hero2.0\\Training Data\\Unprocessed\\04 Paladin - Call of the Night\\notes.mid', '-m']
Error: Executable failed with return code 3762504530
Runnin

Layer sepperated songs

In [29]:
from pydub import AudioSegment
import  os
from pathlib import Path
import gc

#!!!!! NEED CHANGE TO YOUR FFMPEG PATH
os.environ["PATH"] += os.pathsep + r"D:\ffmpeg-2024-05-06-git-96449cfeae-essentials_build\ffmpeg-2024-05-06-git-96449cfeae-essentials_build\bin"

def is_audio_file(filename):
    return filename.lower().endswith(('.ogg', '.mp3', '.wav'))

def layer_tracks(folder_path):
    for dirpath, dirnames, filenames in os.walk(folder_path, topdown=False):
        num_audio_files = 0
        dir_path = Path(dirpath)
        audio_files = []

        for file in filenames:
            if is_audio_file(file):
                num_audio_files += 1
                audio_files.append(dir_path / file)
        if num_audio_files > 1:
            combined = AudioSegment.from_file(str(audio_files[0]))
            print(f"Processing folder: {dirpath} ({num_audio_files} audio files found)")

            try:
                for file in audio_files[1:]:

                    audio_segment = AudioSegment.from_file(str(file))
                    combined = combined.overlay(audio_segment)

                # Export the combined file #misspled on purpose not to be in the pervious list of files
                output_path = dir_path / 'combinned.ogg'

                combined.export(str(output_path), format="ogg")
                print(f"--> Saved combined file to: {output_path}")


                del combined
                del audio_segment

                ##Stop ffmpeg from lingering and stoping the download
                gc.collect()


                #Delete original files
                for file in audio_files:
                    try:
                        os.remove(str(file))
                    except Exception as e:
                        print(f"Error in removing {file}: {e}")


            except Exception as e:
                print(f"Error processing {audio_files}: {e}")


if __name__ == "__main__":
    folder_path = Path(r"D:\tensor-hero2.0\Training Data\Unprocessed")
    layer_tracks(folder_path)

[WindowsPath('D:/temp/Din - Fly on the Wall/guitar.ogg')]
[WindowsPath('D:/temp/Din - Fly on the Wall/guitar.ogg'), WindowsPath('D:/temp/Din - Fly on the Wall/song.ogg')]
Processing folder: D:\temp\Din - Fly on the Wall (2 audio files found)
[WindowsPath('D:/temp/Din - Fly on the Wall/guitar.ogg'), WindowsPath('D:/temp/Din - Fly on the Wall/song.ogg')]
--> Saved combined file to: D:\temp\Din - Fly on the Wall\song.ogg
[WindowsPath('D:/temp/Double Take - Hot Problems [Raspberriel]/guitar.ogg')]
[WindowsPath('D:/temp/Freezepop - Get Ready 2 Rokk/guitar.ogg')]
[WindowsPath('D:/temp/Freezepop - Get Ready 2 Rokk/guitar.ogg'), WindowsPath('D:/temp/Freezepop - Get Ready 2 Rokk/song.ogg')]
Processing folder: D:\temp\Freezepop - Get Ready 2 Rokk (2 audio files found)
[WindowsPath('D:/temp/Freezepop - Get Ready 2 Rokk/guitar.ogg'), WindowsPath('D:/temp/Freezepop - Get Ready 2 Rokk/song.ogg')]
--> Saved combined file to: D:\temp\Freezepop - Get Ready 2 Rokk\song.ogg


Final cleaning pass

In [ ]:
def remove_no_audio_folders(filepath):
    for dirname, dirpath, filelist in os.walk():
        if not filelist.__contains__('.ogg', 'wav' , '.mp3'):
            os.rmdir(dirpath)



Converts to spectrogram and put in new folder

In [2]:
'''
Takes raw, unprocessed data and processes expert .chart files into a specified processed folder
'''

# To process the training data, we will use the populated_processed_folder()
from pathlib import Path
import sys
sys.path.insert(1, str(Path.cwd()))
import os
from tensor_hero.preprocessing.data import populate_processed_folder, populate_with_simplified_notes

repo_root = Path.cwd()
unprocessed_path = repo_root / 'Training Data' / 'Unprocessed'
processed_path = repo_root / 'Training Data' / 'Processed'

assert (os.path.isdir(unprocessed_path)), 'ERROR: Wrong directory: you should have a unprocessed folder in this directory that is already \ncleaned up. make sure to run prior scripts'
if not os.path.isdir(processed_path):
    print("Creating Proccessed folder")
    os.makedirs(processed_path, exist_ok=True)


print('PROCESSING DATA')
processing_data = populate_processed_folder(unprocessed_data_path=unprocessed_path,
                                            processed_data_path=processed_path,
                                            PRINT_TRACEBACK=True, SUB_PACKS=False, TRACK_PACKS=False)
print(f"missing files list: {processing_data}")
print('\nPOPULATING PROCESSED FOLDER WITH SIMPLIFIED NOTES')
populate_with_simplified_notes(processed_path)

AssertionError: ERROR: Wrong directory: you should have a unprocessed folder in this directory that is already 
cleaned up. make sure to run prior scripts

Splits songs into 4 second segmeants

In [1]:
'''
Once 1_preprocess_training_data.py has been run, this script will further process the data
to be ready for the transformer model.
'''

import os
from pathlib import Path
import sys
sys.path.insert(1, str(Path.cwd()))
from tensor_hero.preprocessing.data import preprocess_transformer_data

segment_length = 400
training_data_path = Path(r"D:\tensor-hero2.0\Training Data\Processed")
#training_data_path = Path.cwd() / 'Training_Data' / 'Processed' / 'fully_curated'
train_val_test_probs = [0.95, 0.05, 0]

#folder to put model
model_training_directory_name = 'fully_curated_colab'
full_path = training_data_path / model_training_directory_name

if not os.path.exists(full_path):
    os.mkdir(full_path)


# # If training w/ colab:
# model_training_directory_name = 'colab_training_data'
# preprocess_transformer_data(segment_length,
                            # training_data_path,
                            # train_val_test_probs,
                            # model_training_directory_name,
                            # COLAB=True)

# If not training w/ colab:
# model_training_directory_name = 'colab_training_data'
# preprocess_transformer_data(segment_length,
                            # training_data_path,
                            # train_val_test_probs,
                            # model_training_directory_name,
                            # COLAB=True)

# For generating source separated colab data:
preprocess_transformer_data(segment_length,
                            training_data_path,
                            train_val_test_probs,
                            model_training_directory_name,
                            COLAB=True,
                            SEPARATED=False)

0it [00:00, ?it/s]


Next Go to the Model folder and run one of the training files